In [0]:
# Importing Packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import h5py as hf
import time

In [0]:
# SciKit Imports
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.neighbors import KNeighborsClassifier

In [0]:
# Keras Imports

from keras.models import Sequential

from keras.layers import Dense
from keras.layers import BatchNormalization
from keras.layers import Dropout

from keras.optimizers import adam
from keras.optimizers import SGD

from keras.utils import to_categorical
from keras.wrappers.scikit_learn import KerasClassifier

In [4]:
# This step is only for running it on google colab
from google.colab import drive
drive.mount('/content/gdrive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=email%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocs.test%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.photos.readonly%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fpeopleapi.readonly&response_type=code

Enter your authorization code:
··········
Mounted at /content/gdrive


In [0]:
# The file is stored in google drive
file = hf.File('/content/gdrive/My Drive/SVHN_single_grey1.h5','r')

In [13]:
list(file.keys())

['X_test', 'X_train', 'X_val', 'y_test', 'y_train', 'y_val']

In [14]:
file['X_test'].shape , file['X_train'].shape, file['X_val'].shape

((18000, 32, 32), (42000, 32, 32), (60000, 32, 32))

In [15]:
file['y_test'].shape , file['y_train'].shape, file['y_val'].shape

((18000,), (42000,), (60000,))

In [16]:
32*32

1024

In [17]:
# Converting from h5 to numpy arrays

X_train = np.array(file['X_train'])
X_test = np.array(file['X_test'])
y_train = np.array(file['y_train'])
y_test = np.array(file['y_test'])

X_train.shape,y_train.shape,X_test.shape,y_test.shape

((42000, 32, 32), (42000,), (18000, 32, 32), (18000,))

In [0]:
# Normalising the values

X_train = X_train/255
X_test = X_test/255

In [19]:
# OneHotEncoding of target y

y_train_cat = to_categorical(y_train,num_classes=10)
y_test_cat = to_categorical(y_test,num_classes=10)

y_train_cat.shape,y_test_cat.shape

((42000, 10), (18000, 10))

In [20]:
X_train.shape, X_test.shape

((42000, 32, 32), (18000, 32, 32))

In [0]:
# Reshape the inputs 
X_train = X_train.reshape(42000,1024)
X_test = X_test.reshape(18000,1024)

In [22]:
X_train.shape, X_test.shape

((42000, 1024), (18000, 1024))

# **Using KNN**

In [0]:
knn = KNeighborsClassifier(n_neighbors=20)

In [17]:
knn.fit(X_train,y_train_cat)

KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=20, p=2,
           weights='uniform')

**Note this step takes over 40 minutes to compute !!!**

In [18]:
#Predict

start = time. time()
pred = knn.predict(X_test)
end = time. time()
print('Note this takes a long time to compute',end - start)

Note this takes a long time to compute 1781.858998298645


In [0]:
# Selecting the categorty to be the one with highest probablity

y_pred_class = np.argmax(pred, axis=1)

In [20]:
# Calculating Accuracy of the model 

accuracy_score(y_test, y_pred_class)

0.23027777777777778

## **KNN gives us an accuracy of 23%, In the next step we check the accuracy obtained by a NN**

In [0]:
print(confusion_matrix(y_test,y_pred_class))

[[1810    2    0    0    0    0    0    0    0    2]
 [1333  491    0    0    2    1    0    1    0    0]
 [1527    4  267    1    1    0    0    3    0    0]
 [1617    5    2   90    2    2    0    0    1    0]
 [1298   15    0    0  499    0    0    0    0    0]
 [1663    4    0    1    1   98    1    0    0    0]
 [1727    2    0    0    3    0   97    0    3    0]
 [1228   23    3    0    0    0    0  554    0    0]
 [1739    2    0    0    0    0    0    0   71    0]
 [1628    3    0    1    1    0    0    3    0  168]]


In [21]:
print(classification_report(y_test,y_pred_class))

              precision    recall  f1-score   support

           0       0.12      1.00      0.21      1814
           1       0.89      0.27      0.41      1828
           2       0.98      0.15      0.26      1803
           3       0.97      0.05      0.10      1719
           4       0.98      0.28      0.43      1812
           5       0.97      0.06      0.10      1768
           6       0.99      0.05      0.10      1832
           7       0.99      0.31      0.47      1808
           8       0.95      0.04      0.08      1812
           9       0.99      0.09      0.17      1804

   micro avg       0.23      0.23      0.23     18000
   macro avg       0.88      0.23      0.23     18000
weighted avg       0.88      0.23      0.23     18000



# **Using Neural Networks**

In [26]:
# Instantiate the model
model = Sequential()

# Add a Dense Layer with 32 neurons
model.add(Dense(32,input_shape=(1024,),activation='relu'))
model.add(BatchNormalization())
model.add(Dense(512,activation='relu'))
model.add(Dropout(0.5)) # Drop out layer to reduce over fitting
model.add(Dense(1024,activation='relu'))
model.add(Dense(1024,activation='relu'))
model.add(Dense(1024,activation='relu'))

# Output layer with 10 neurons and softmax and it a classification problem 
model.add(Dense(10,activation='softmax'))

Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


In [27]:
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_1 (Dense)              (None, 32)                32800     
_________________________________________________________________
batch_normalization_1 (Batch (None, 32)                128       
_________________________________________________________________
dense_2 (Dense)              (None, 512)               16896     
_________________________________________________________________
dropout_1 (Dropout)          (None, 512)               0         
_________________________________________________________________
dense_3 (Dense)              (None, 1024)              525312    
_________________________________________________________________
dense_4 (Dense)              (None, 1024)              1049600   
_________________________________________________________________
dense_5 (Dense)              (None, 1024)              1049600   
__________

In [0]:
sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(optimizer=sgd,loss='categorical_crossentropy',metrics=['accuracy'])

**This step takes over 60 minutes to compute**

In [34]:
model.fit(X_train,
          y_train_cat,
          batch_size=64,
          epochs=100,
          verbose=1,
          validation_split=0.3)

Train on 29399 samples, validate on 12601 samples
Epoch 1/100
29399/29399 [==============================] - 32s 1ms/step - loss: 0.4803 - acc: 0.8386 - val_loss: 0.6179 - val_acc: 0.8183
Epoch 2/100
29399/29399 [==============================] - 33s 1ms/step - loss: 0.4790 - acc: 0.8380 - val_loss: 0.6775 - val_acc: 0.7955
Epoch 3/100
29399/29399 [==============================] - 30s 1ms/step - loss: 0.4758 - acc: 0.8399 - val_loss: 0.6966 - val_acc: 0.7915
Epoch 4/100
29399/29399 [==============================] - 31s 1ms/step - loss: 0.4676 - acc: 0.8403 - val_loss: 0.7123 - val_acc: 0.7857
Epoch 5/100
29399/29399 [==============================] - 31s 1ms/step - loss: 0.4618 - acc: 0.8452 - val_loss: 0.6308 - val_acc: 0.8178
Epoch 6/100
29399/29399 [==============================] - 31s 1ms/step - loss: 0.4635 - acc: 0.8416 - val_loss: 0.6364 - val_acc: 0.8124
Epoch 7/100
29399/29399 [==============================] - 32s 1ms/step - loss: 0.4555 - acc: 0.8436 - val_loss: 0.7294 - 

In [0]:
# Predicting test values
y_pred = model.predict(X_test)

In [0]:
# Selecting the categorty to be the one with highest probablity

y_pred_class = np.argmax(y_pred, axis=1)

In [37]:
# Calculating Accuracy of the model 

accuracy_score(y_test, y_pred_class)

0.7834444444444445

## **NN Model has a training accuracy of 93% and test data accuracy of 78.3%, which is far higher than the 23% accuracy obtained with KNN**